<a href="https://colab.research.google.com/github/BrundaSreedhar/credit-card-fraud-detection/blob/main/RandomForestExperiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Initial EDA

In [1]:
import pandas as pd
import numpy as np

# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "creditcard.csv"

# Load the latest version
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
  "mlg-ulb/creditcardfraud",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:")
print(df.head())
print(df.shape)

Using Colab cache for faster access to the 'creditcardfraud' dataset.
First 5 records:
   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ...

#Data Preprocessing


In [2]:
#Amount ranges from 0 to 25691.160000

from sklearn.preprocessing import RobustScaler, StandardScaler
new_df = df.copy()
new_df['Amount'] = RobustScaler().fit_transform(new_df['Amount'].to_numpy().reshape(-1, 1))

In [3]:
time = new_df['Time']
#standard scaler
new_df['Time'] = StandardScaler().fit_transform(new_df[['Time']])

In [4]:
new_df = new_df.sample(frac=1, random_state=42)

In [5]:
from sklearn.model_selection import train_test_split

train, temp = train_test_split(
    new_df,
    test_size=0.2,
    stratify=new_df['Class'],
    random_state=42
)

test, val = train_test_split(
    temp,
    test_size=0.5,
    stratify=temp['Class'],
    random_state=42
)

In [6]:
x_train = train.drop(columns=['Class'])
y_train = train['Class']

x_test = test.drop(columns=['Class'])
y_test = test['Class']

x_val = val.drop(columns=['Class'])
y_val = val['Class']

x_train.shape, y_train.shape, x_test.shape, y_test.shape, x_val.shape, y_val.shape

((227845, 30), (227845,), (28481, 30), (28481,), (28481, 30), (28481,))

#Modelling

#Random Forest

Experiments with and without SMOTE

In [8]:
#Random forest to test the data
from sklearn.metrics import precision_recall_curve, classification_report, average_precision_score
from sklearn.ensemble import RandomForestClassifier

random_forest = RandomForestClassifier()
random_forest.fit(x_train, y_train)

print(classification_report(y_val, random_forest.predict(x_val), target_names=['Legit', 'Fraud']))

              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     28432
       Fraud       0.90      0.78      0.84        49

    accuracy                           1.00     28481
   macro avg       0.95      0.89      0.92     28481
weighted avg       1.00      1.00      1.00     28481



In [9]:
#try tuning threshold

y_probs = random_forest.predict_proba(x_val)[:, 1]   # probability of fraud
precision, recall, thresholds = precision_recall_curve(y_val, y_probs)

#choose threshold that maximizes F1
f1 = 2 * (precision * recall) / (precision + recall + 1e-9)
best_idx = np.argmax(f1)
best_threshold = thresholds[best_idx]

print("Best threshold:", best_threshold)

print("After tuning threshold.. ")
y_pred_tuned = (y_probs >= best_threshold).astype(int)
print(classification_report(y_val, y_pred_tuned))
print("PR AUC ", average_precision_score(y_val, y_probs))

Best threshold: 0.29
After tuning threshold.. 
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     28432
           1       0.87      0.84      0.85        49

    accuracy                           1.00     28481
   macro avg       0.94      0.92      0.93     28481
weighted avg       1.00      1.00      1.00     28481

PR AUC  0.8307390477736118


In [ ]:
#get important features and train model on the important features alone


Calibrated Classfier CV

In [ ]:
"""
from sklearn.calibration import CalibratedClassifierCV

calibrated_rf = CalibratedClassifierCV(
    random_forest,
    method='sigmoid',
    cv='prefit'
)

calibrated_rf.fit(x_val, y_val)

y_probs = calibrated_rf.predict_proba(x_val)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_val, y_probs)
#choose threshold that maximizes F1
f1 = 2 * (precision * recall) / (precision + recall + 1e-9)
best_idx = np.argmax(f1)
best_threshold = thresholds[best_idx]

print("Best threshold:", best_threshold)

print("After tuning threshold.. ")
y_pred_tuned = (y_probs >= best_threshold).astype(int)
print(classification_report(y_val, y_pred_tuned))
ap = average_precision_score(y_val, y_probs)
print("PR-AUC:", ap)
"""

/usr/local/lib/python3.12/dist-packages/sklearn/calibration.py:333: UserWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


Best threshold: 0.026154792518285626
After tuning threshold.. 
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     28432
           1       0.91      0.84      0.87        49

    accuracy                           1.00     28481
   macro avg       0.96      0.92      0.94     28481
weighted avg       1.00      1.00      1.00     28481

PR-AUC: 0.8272119885020576


this classifier has pretty good precision, but we'd like to see if we could improve the recall without destroying precision.


Lets try adding class weights:

In [10]:
random_forest_weighted = RandomForestClassifier(class_weight="balanced")
random_forest_weighted.fit(x_train, y_train)

weighted_predictions = random_forest_weighted.predict(x_val)
print(classification_report(y_val, weighted_predictions, target_names=['Legit', 'Fraud']))

              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     28432
       Fraud       0.95      0.71      0.81        49

    accuracy                           1.00     28481
   macro avg       0.97      0.86      0.91     28481
weighted avg       1.00      1.00      1.00     28481



In [11]:
#tune threshold

precision, recall, thresholds = precision_recall_curve(y_val, weighted_predictions)
#choose threshold that maximizes F1
f1 = 2 * (precision * recall) / (precision + recall + 1e-9)
best_idx = np.argmax(f1)
best_threshold = thresholds[best_idx]

print("Best threshold:", best_threshold)

print("After tuning threshold.. ")
y_pred_tuned = (y_probs >= best_threshold).astype(int)
print(classification_report(y_val, weighted_predictions))
ap = average_precision_score(y_val, weighted_predictions)
print("PR-AUC:", ap)

Best threshold: 1
After tuning threshold.. 
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     28432
           1       0.95      0.71      0.81        49

    accuracy                           1.00     28481
   macro avg       0.97      0.86      0.91     28481
weighted avg       1.00      1.00      1.00     28481

PR-AUC: 0.6761672314497005


In [29]:
#get the most important features from the model
print("Unweighted...")
feature_importances = random_forest.feature_importances_
feature_names = x_train.columns
sorted_indices = np.argsort(feature_importances)[::-1]
print("Top 10 important features:")
for i in range(len(x_train.columns)):
    print(f"{feature_names[sorted_indices[i]]}: {feature_importances[sorted_indices[i]]}")


Unweighted...
Top 10 important features:
V17: 0.17244351385403747
V12: 0.11867502593051837
V14: 0.11573269147782918
V11: 0.09523875140807593
V10: 0.08750290354566109
V16: 0.05740446303060061
V9: 0.03213843305436695
V18: 0.030708066295548814
V4: 0.029056224359310115
V7: 0.022953020251599256
V26: 0.0189881515037496
V3: 0.018519672047350434
V21: 0.01828276462365673
V20: 0.013357167951373395
V1: 0.013206188182133129
V6: 0.012699560939401862
Time: 0.012402456588886399
V27: 0.011970019939435455
V19: 0.011465544919251135
V8: 0.011251816856766102
V24: 0.011238160086847648
V22: 0.010703047077855339
Amount: 0.010594865015533392
V2: 0.010246037505282149
V15: 0.010163517730974738
V13: 0.010069420383516912
V5: 0.009899204171990751
V25: 0.009207334473907935
V28: 0.00878442596332151
V23: 0.005097550831217551


In [30]:
#keep just the important features and train the model on them
important_features = feature_names[sorted_indices[:20]]
x_train_important = x_train[important_features]
x_val_important = x_val[important_features]

random_forest_important = RandomForestClassifier(random_state=42)
random_forest_important.fit(x_train_important, y_train)

y_important_predictions = random_forest_important.predict(x_val_important)
print(classification_report(y_val, y_important_predictions, target_names=['Legit', 'Fraud']))



              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     28432
       Fraud       0.90      0.78      0.84        49

    accuracy                           1.00     28481
   macro avg       0.95      0.89      0.92     28481
weighted avg       1.00      1.00      1.00     28481



In [31]:
#train a regular RF with just the features, random_state = 42

random_forest_unweighted_important = RandomForestClassifier(random_state=42)
random_forest_unweighted_important.fit(x_train_important, y_train)

y_important_predictions_unweighted = random_forest_unweighted_important.predict(x_val_important)
print(classification_report(y_val, y_important_predictions_unweighted, target_names=['Legit', 'Fraud']))


              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     28432
       Fraud       0.90      0.78      0.84        49

    accuracy                           1.00     28481
   macro avg       0.95      0.89      0.92     28481
weighted avg       1.00      1.00      1.00     28481



In [32]:
#threshold tune the results
y_probs = random_forest_unweighted_important.predict_proba(x_val_important)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_val, y_probs)

f1 = 2 * (precision * recall) / (precision + recall + 1e-9)
best_idx = np.argmax(f1)
best_threshold = thresholds[best_idx]

#adjust labels
y_pred_tuned = (y_probs >= best_threshold).astype(int)
print(classification_report(y_val, y_pred_tuned))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00     28432
           1       0.91      0.84      0.87        49

    accuracy                           1.00     28481
   macro avg       0.96      0.92      0.94     28481
weighted avg       1.00      1.00      1.00     28481



Let's try SMOTE + Random Forest


In [22]:
pip install imblearn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.4/235.4 kB 15.0 MB/s eta 0:00:00


In [33]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# define pipeline
pipeline = Pipeline([
    ('smote', SMOTE(random_state=42 )),
    ('rf', RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_leaf=4,
        n_jobs=-1,
        random_state=42
    ))
])

# train (SMOTE applied ONLY to training data)
pipeline.fit(x_train_important, y_train)

# predict
y_pred_smote = pipeline.predict(x_val_important)

print(classification_report(y_val, y_pred_smote))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     28432
           1       0.84      0.84      0.84        49

    accuracy                           1.00     28481
   macro avg       0.92      0.92      0.92     28481
weighted avg       1.00      1.00      1.00     28481



In [34]:

y_probs = pipeline.predict_proba(x_val_important)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_val, y_probs)

#choose threshold that maximizes F1
f1 = 2 * (precision * recall) / (precision + recall + 1e-9)
best_idx = np.argmax(f1)
best_threshold = thresholds[best_idx]

print("Best threshold:", best_threshold)

print("After tuning threshold for RF+SMOTE with reduced dimensions.. ")
y_pred_tuned = (y_probs >= best_threshold).astype(int)
print(classification_report(y_val, y_pred_tuned))

ap = average_precision_score(y_val, y_probs)
print("PR-AUC:", ap)
print("old precision was 0.8 for fraud with precision 0.87!")

Best threshold: 0.6598111471861472
After tuning threshold for RF+SMOTE with reduced dimensions.. 
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     28432
           1       0.91      0.82      0.86        49

    accuracy                           1.00     28481
   macro avg       0.95      0.91      0.93     28481
weighted avg       1.00      1.00      1.00     28481

PR-AUC: 0.8095609553454878
old precision was 0.8 for fraud with precision 0.87!


Another experiment with RF + SMOTE

In [35]:
from sklearn.metrics import precision_recall_curve, classification_report, average_precision_score
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier


rf_model = Pipeline([
    ("smote", SMOTE(
        sampling_strategy=0.4,   # not full balance → to prevent overfitting
        k_neighbors=5,
        random_state=42
    )),
    ("rf", RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_leaf=4,
        n_jobs=-1,
        random_state=42
    ))
])

#fit model
rf_model.fit(x_train_important, y_train)

y_probs = rf_model.predict_proba(x_val_important)[:, 1]
#threshold tuning
precision, recall, thresholds = precision_recall_curve(y_val, y_probs)
#print metrics before threshold tuning
print("Metrics before threshold tuning:")
print(classification_report(y_val, (y_probs >= 0.7).astype(int)))

#tune threshold for best recall


f1 = 2 * precision * recall / (precision + recall + 1e-9)
best_idx = np.argmax(f1)
best_threshold = thresholds[best_idx]

print("Best threshold:", best_threshold)

y_pred = (y_probs >= best_threshold).astype(int)

print(classification_report(y_val, y_pred))

pr_auc = average_precision_score(y_val, y_probs)
print("PR-AUC:", pr_auc)
print("old precision 0.93 and recall 0.78")

Metrics before threshold tuning:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     28432
           1       0.91      0.82      0.86        49

    accuracy                           1.00     28481
   macro avg       0.95      0.91      0.93     28481
weighted avg       1.00      1.00      1.00     28481

Best threshold: 0.6070417637917638
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     28432
           1       0.91      0.84      0.87        49

    accuracy                           1.00     28481
   macro avg       0.96      0.92      0.94     28481
weighted avg       1.00      1.00      1.00     28481

PR-AUC: 0.8115619499692914
old precision 0.93 and recall 0.78


0
Best threshold for 90% precision: 0.0
              precision    recall  f1-score   support

           0       0.00      0.00      0.00     28432
           1       0.00      1.00      0.00        49

    accuracy                           0.00     28481
   macro avg       0.00      0.50      0.00     28481
weighted avg       0.00      0.00      0.00     28481



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Cross-validated training

In [ ]:
param_dist = {
    "smote__sampling_strategy": [0.2, 0.5, 0.3],  # how much to oversample
    "smote__k_neighbors": [3, 5, 7],

    "rf__n_estimators": [100, 200, 300],
    "rf__max_depth": [6, 8, 10, None],
    "rf__min_samples_leaf": [1, 2, 4]
}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_dist,
    n_iter=15,
    scoring="average_precision",   # PR-AUC for imbalanced data
    cv=3,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

search.fit(x_train, y_train)

Fitting 3 folds for each of 15 candidates, totalling 45 fits


RandomizedSearchCV(cv=3,
                   estimator=Pipeline(steps=[('smote', SMOTE(random_state=42)),
                                             ('rf',
                                              RandomForestClassifier(min_samples_leaf=4,
                                                                     n_estimators=200,
                                                                     n_jobs=-1,
                                                                     random_state=42))]),
                   n_iter=15, n_jobs=-1,
                   param_distributions={'rf__max_depth': [6, 8, 10, None],
                                        'rf__min_samples_leaf': [1, 2, 4],
                                        'rf__n_estimators': [100, 200, 300],
                                        'smote__k_neighbors': [3, 5, 7],
                                        'smote__sampling_strategy': [0.2, 0.5,
                                                                     0.3]},
                   random_state=42, scoring='average_precision', verbose=1)

In [ ]:
print("Best PR-AUC:", search.best_score_)
print("Best params:", search.best_params_)
best_model = search.best_estimator_

Best PR-AUC: 0.8594185205450261
Best params: {'smote__sampling_strategy': 0.3, 'smote__k_neighbors': 5, 'rf__n_estimators': 300, 'rf__min_samples_leaf': 1, 'rf__max_depth': None}


In [ ]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier

final_model = Pipeline([
    ("smote", SMOTE(
        sampling_strategy=0.3,
        k_neighbors=5,
        random_state=42
    )),
    ("rf", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1
    ))
])

In [ ]:
final_model.fit(x_train, y_train)

Pipeline(steps=[('smote', SMOTE(random_state=42, sampling_strategy=0.3)),
                ('rf',
                 RandomForestClassifier(n_estimators=300, n_jobs=-1,
                                        random_state=42))])

In [ ]:
from sklearn.metrics import classification_report, average_precision_score

y_pred = final_model.predict(x_val)
y_probs = final_model.predict_proba(x_val)[:, 1]
print(classification_report(y_val, y_pred))
print("PR-AUC:", average_precision_score(y_val, y_probs))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00     28432
           1       0.89      0.82      0.85        49

    accuracy                           1.00     28481
   macro avg       0.94      0.91      0.93     28481
weighted avg       1.00      1.00      1.00     28481

PR-AUC: 0.8239680295613971


In [37]:
#search for the lowest threshold that gives you at least 90% precision

precisions, recalls, thresholds = precision_recall_curve(y_val, y_probs)

idx_for_90_recall = (recalls >= 0.88).argmax()
threshold_for_90_precision = thresholds[idx_for_90_recall]

print("Best threshold for 90% precision:", threshold_for_90_precision)
y_val_pred_90 = (y_probs >= threshold_for_90_precision)
print(classification_report(y_val, y_val_pred_90))


Best threshold for 90% precision: 0.0
              precision    recall  f1-score   support

           0       0.00      0.00      0.00     28432
           1       0.00      1.00      0.00        49

    accuracy                           0.00     28481
   macro avg       0.00      0.50      0.00     28481
weighted avg       0.00      0.00      0.00     28481



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
#gridsearchCV

from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)

param_dist = {
    "n_estimators": [100, 200, 300],
    "max_depth": [6, 8, 10, None],
    "min_samples_leaf": [1, 2, 4, 8]
}

search = RandomizedSearchCV(
    rf,
    param_distributions=param_dist,
    n_iter=10,                   # ⭐ only try 10 random combos
    scoring="average_precision",
    cv=3,
    n_jobs=-1,
    random_state=42
)

search.fit(x_train, y_train)

print("Best PR-AUC:", search.best_score_)
print("Best params:", search.best_params_)

print("Best Estimator", search.best_estimator_)

Best PR-AUC: 0.8452191679323698
Best params: {'n_estimators': 200, 'min_samples_leaf': 4, 'max_depth': None}
Best Estimator RandomForestClassifier(min_samples_leaf=4, n_estimators=200, random_state=42)
